In [ ]:
# packages
import numpy as np 
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.pyplot import subplots
import statsmodels.api as sm
from ISLP import (load_data, confusion_table)
from ISLP.models import (ModelSpec as MS, summarize, contrast)
from sklearn.model_selection import train_test_split 
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import RocCurveDisplay, roc_auc_score

roc_curve_est = RocCurveDisplay.from_estimator 
roc_curve_pred = RocCurveDisplay.from_predictions 

# set seed
seed = 5331

### We will use the OJ dataset, which contains information about orange juice purchases across different stores.

In [ ]:
OJ = load_data('OJ')
OJ

### What are the variables and their types?

In [ ]:
OJ.dtypes

### We will use all stores other than ID=7 as training data, and Store #7 will be used as test.

In [ ]:
OJ["StoreID"].value_counts()

In [ ]:
Train = OJ[OJ['StoreID'] != 7]
Train

In [ ]:
Test = OJ[OJ['StoreID']==7]

### We will predict whether each customer purchased Citrus Hill or Minute Maid.

In [ ]:
y_train = Train.Purchase == 'CH'
y_test = Test.Purchase == 'CH'

In [ ]:
X_train = Train[['WeekofPurchase','PriceCH','PriceMM','DiscCH','DiscMM','SpecialCH','SpecialMM','LoyalCH','SalePriceCH','SalePriceMM','PriceDiff','PctDiscCH','PctDiscMM','ListPriceDiff']]
X_test = Test[['WeekofPurchase','PriceCH','PriceMM','DiscCH','DiscMM','SpecialCH','SpecialMM','LoyalCH','SalePriceCH','SalePriceMM','PriceDiff','PctDiscCH','PctDiscMM','ListPriceDiff']]

In [ ]:
X_train['intercept'] = np.ones(X_train.shape[0])
X_test['intercept'] = np.ones(X_test.shape[0])

## Logistic Regression

In [ ]:
var_list = ['intercept','LoyalCH','PriceDiff']

glm = sm.GLM(y_train,
             X_train[var_list],
             family=sm.families.Binomial())

results = glm.fit()

summarize(results)

In [ ]:
def predict(X, model):
    predictions_df = pd.DataFrame(model.get_prediction(X).predicted,
                                  columns=['y_hat'],
                                  index=X.index)
    return predictions_df['y_hat']

In [ ]:
probs_train=predict(X_train[var_list],results)
probs_test=predict(X_test[var_list],results)

In [ ]:
predictions_train = np.array([True]*len(y_train))
predictions_train[probs_train<0.5] = False

predictions_test = np.array([True]*len(y_test))
predictions_test[probs_test<0.5] = False

In [ ]:
train_table = confusion_table(predictions_train, y_train)
train_table

In [ ]:
test_table = confusion_table(predictions_test, y_test)
test_table

In [ ]:
false_positive_rate = np.sum((predictions_test == True) & (y_test == False)) / np.sum(y_test == False)
print("fpr =",false_positive_rate)

false_negative_rate = np.sum((predictions_test == False) & (y_test == True)) / np.sum(y_test == True)
print("fnr =",false_negative_rate)

In [ ]:
var_list.remove('intercept')
X_train_array, X_test_array = [np.asarray(X) for X in [X_train[var_list], X_test[var_list]]]

In [ ]:
best_k = 0
num_correct_pred = 0

for k in range(1,500):
    knn = KNeighborsClassifier(n_neighbors=k)
    knn.fit(X_train_array, y_train)
    knn_pred = knn.predict(X_test_array)
    correct = np.sum(knn_pred == y_test)

    if correct > num_correct_pred:
        num_correct_pred = correct
        best_k = k

print(best_k)
print(num_correct_pred)

In [ ]:
knn_opt = KNeighborsClassifier(n_neighbors=best_k)
knn_opt.fit(X_train_array, y_train)
knn_opt_test = knn_opt.predict(X_test_array)

confusion_table(knn_opt_test, y_test)

### Of the models that were built in this notebook, which would you choose to implement? Why?

I would choose the logistic regression model. It is simple, easy to interpret, and uses only the most important predictors (loyalty and price difference), which helps reduce overfitting. Logistic regression also provides predicted probabilities, which makes it easier to understand the model’s confidence and adjust the classification threshold if needed.

In [ ]:
misclassification_rate = np.sum(y_test == False) / len(y_test)
print(misclassification_rate)